In [1]:
import ipdb # <- трасировка и точки останова
import header
from header import __root__
from src import gs

🔑 Found password in password.txt (DEBUG MODE)
✅ Successfully opened KeePass database: C:\Users\user\Documents\repos\hypotez\secrets\credentials.kdbx
Failed to extract Aliexpress API key from KeePass 'types.SimpleNamespace' object has no attribute 'aliexpress_com'
Failed to load Aliexpress credentials
Failed to load GAPI credentials


In [2]:
import asyncio
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, Dict, Any, List

from src.webdriver.pydoll.llib.pydoll.browser.chrome import Chrome
from src.webdriver.pydoll.llib.pydoll.constants import By

from src.llm.gemini import GoogleGenerativeAi # Unused, but kept
from src.endpoints.prestashop.product_fields import ProductFields

from src.endpoints.prestashop.product_async import PrestaProductAsync
from src.endpoints.prestashop.product import PrestaProduct

from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory
from src.utils.jjson import j_loads, j_loads_ns, j_dumps # j_dumps unused
from src.utils.image import get_image_bytes, get_raw_image_data 
from src.utils.printer import pprint as print
from src.logger.logger import logger

In [3]:
class Config:
    """Класс конфигурации скрипта."""
    ENDPOINT: Path = __root__ / 'SANDBOX' / 'davidka'
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'
    SCENARIOS_DIR: Path = __root__ / 'SANDBOX' / 'davidka' / 'scenarios'
    # config: SimpleNamespace = j_loads_ns(ENDPOINT / 'davidka.json') #  general config.
    scenarios_files: List[str] = get_filenames_from_directory(SCENARIOS_DIR) # SANDBOX/davidka/scenarios/*.json
    PRESTA_API_KEY: str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_API_DOMAIN: str = gs.credentials.prestashop.store_davidka_net.api_domain
    #presta_product: PrestaProductAsync = PrestaProductAsync(api_key=PRESTA_API_KEY, api_domain=PRESTA_API_DOMAIN)

In [4]:
browser:Chrome = None
page:'Page' = None

In [5]:
supplier_prefix:str = 'morlevi.co.il'
supplier_alias:str = supplier_prefix.replace('.','_').replace('-','_')
supplier_config_path:Path = Config.SUPPLIERS_ENDPOINT / supplier_alias
locators_path:Path = supplier_config_path / 'locators'
product_locators:SimpleNamespace = j_loads_ns(locators_path / 'product.json')
category_locators:SimpleNamespace = j_loads_ns(locators_path / 'category.json')
product_url = fr'https://www.morlevi.co.il/product/19779'


In [6]:
async def run_scenario():
    """
    Исполнять сценарии лучше по такому шаблону"""
    
    async with Chrome() as browser:
        await browser.start()
        page = await browser.get_page()
        await page.go_to(product_url)
    

In [7]:
if not browser:
    browser = Chrome()  
    await browser.start()
    
if not page:
    page = await browser.get_page()
        

2025-07-20 17:55:58,880 - INFO - EventsHandler initialized
2025-07-20 17:55:58,882 - INFO - ConnectionHandler initialized.
2025-07-20 17:55:59,212 - INFO - Connecting to ws://localhost:9317/devtools/browser/1c995b31-d8f6-46fb-b3ed-2bbe20fada1c
2025-07-20 17:56:01,267 - INFO - EventsHandler initialized
2025-07-20 17:56:01,268 - INFO - ConnectionHandler initialized.


In [8]:
await page.go_to(product_url)

2025-07-20 17:56:05,807 - INFO - Connecting to ws://localhost:9317/devtools/page/9B0608A029D7B4AAD8C08DC2B5FDA3C5


In [ ]:
# strategy: Dict[str, By] = {
#     'XPATH': By.XPATH,
#     'CSS_SELECTOR': By.CSS_SELECTOR,
# }

In [26]:
async def execute_locator(page,  locator: SimpleNamespace):
    """Locate and return content from the element based on locator info."""
    _webelement = await page.find_element(By[locator.by.upper()], locator.selector)
    #ipdb.set_trace()
    match locator.attribute.lower():
        case 'innertext':
            return await _webelement.get_element_text()
        case 'innerhtml':
            return await _webelement.inner_html
        case 'src'|'href':
            return _webelement.get_attribute(locator.attribute.lower())
    # Можно добавить return None или raise, если атрибут неизвестен

In [10]:
product_locators:SimpleNamespace = j_loads_ns(locators_path / 'product.json') # Обновить после редакции JSON 

In [11]:
f:ProductFields = ProductFields()

In [12]:
print(product_locators.name)

namespace(attribute='innerText', by='XPATH', strategy_for_multiple_selectors='find_first_match', selector="//div[contains(@class, 'product-header')]//h1", if_list='first', timeout=0, timeout_for_event='presence_of_element_located', event=None, mandatory=True, locator_description='morlevi - `name`')


In [13]:
f.name = await execute_locator(page, product_locators.name)

In [14]:
print(f.name)

{
    "language": {
        "attrs": {
            "id": 1
        },
        "value": "\u05d6\u05db\u05e8\u05d5\u05df \u05dc\u05e0\u05d9\u05d9\u05d7 Netac 8GB DDR3 1600MHZ CL11"
    }
}


In [ ]:
f.price = await execute_locator(page, product_locators.price)

In [ ]:
f.description =  await execute_locator(page, product_locators.description)

In [21]:
f.specification = await execute_locator(page, product_locators.specification)

In [27]:
f.default_image_url =   await execute_locator(page, product_locators.default_image_url)

In [28]:
print(f.default_image_url)

/cache/w_1000/NAC-NTBSD3P16SP-08-30245163-01.png


2025-07-20 19:52:26,196 - INFO - Connection closed gracefully: no close frame received or sent


In [25]:
print(f.default_image_url)

In [ ]:
# ПРАВИЛЬНО:
async with PrestaProductAsync(api_key=Config.PRESTA_API_KEY, api_domain=Config.PRESTA_API_DOMAIN) as presta_product_api:
    # Теперь presta_product_api.client инициализирован
    result = await presta_product_api.add_new_product_async(f)

In [ ]:
fields = {
    'name': await page.find_element(strategy[locator_product.name.by], locator_product.name.selector),
    'price': await page.find_element(strategy[locator_product.price.by], locator_product.price.selector),
    'id_supplier': locator_product.id_supplier.attr, 
    'description_short': await page.find_element(strategy[locator_product.description_short.by], locator_product.description_short.selector),
    'description': await page.find_element(strategy[locator_product.description.by], locator_product.description.selector),
    'specification': await page.find_element(strategy[locator_product.specification.by], locator_product.specification.selector),
    'default_image_url': await page.find_element(strategy[locator_product.default_image_url.by], locator_product.default_image_url.selector),
}